# Federated Learning Fundamentals: FedAvg

Federated learning (McMahan et al. 2017) trains models across decentralized data without moving the data to a central server. The canonical algorithm — FedAvg — averages model updates from participating clients.

## The Federated Learning Setup

Classical ML assumes all data is centralized. FL relaxes this:
- Data stays on N clients (mobile devices, hospitals, banks)
- Each client trains on its local data and sends weight updates (not data) to a server
- The server aggregates updates and distributes a new global model
- Repeat until convergence

**FedAvg** aggregates by weighted averaging: each client's update is weighted by its dataset size, so larger clients contribute more. Communication happens in rounds — not continuously.

**Non-IID challenge**: in real federated settings, client data distributions differ significantly. A hospital in one city has different patient demographics than one in another. Non-IID data causes FedAvg to converge slowly or to a suboptimal solution.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable
import random, math

@dataclass
class FLClient:
    client_id: int
    n_samples: int
    data_mean: float
    data_std: float
    local_model: list = field(default_factory=lambda: [0.0])

@dataclass
class FLRound:
    round_num: int
    n_clients: int
    global_model: list
    avg_local_loss: float
    communication_bytes: int

class FedAvgSimulator:
    def __init__(self, clients: list, global_model: list, n_rounds: int = 20,
                 fraction_clients: float = 0.5, local_epochs: int = 5, lr: float = 0.01):
        self.clients = clients
        self.global_model = list(global_model)
        self.n_rounds = n_rounds
        self.fraction = fraction_clients
        self.local_epochs = local_epochs
        self.lr = lr
        self.history: list = []
        self.rng = random.Random(42)

    def _local_train(self, client: FLClient) -> tuple:
        model = list(self.global_model)
        local_loss = 0.0
        for _ in range(self.local_epochs):
            batch = [self.rng.gauss(client.data_mean, client.data_std) for _ in range(32)]
            grad = sum(2*(model[0] - x) for x in batch) / len(batch)
            model[0] -= self.lr * grad
            local_loss += sum((model[0]-x)**2 for x in batch) / len(batch)
        client.local_model = model
        return model, local_loss / self.local_epochs

    def _fedavg(self, updates: list, weights: list) -> list:
        total = sum(weights)
        n_params = len(updates[0])
        avg = [sum(u[i]*w for u, w in zip(updates, weights))/total for i in range(n_params)]
        return avg

    def run(self) -> list:
        for round_num in range(self.n_rounds):
            n_selected = max(1, int(len(self.clients) * self.fraction))
            selected = self.rng.sample(self.clients, n_selected)
            updates, losses, weights = [], [], []
            for client in selected:
                update, loss = self._local_train(client)
                updates.append(update)
                losses.append(loss)
                weights.append(client.n_samples)
            self.global_model = self._fedavg(updates, weights)
            bytes_communicated = n_selected * len(self.global_model) * 4 * 2  # up+down
            self.history.append(FLRound(
                round_num=round_num+1, n_clients=n_selected,
                global_model=list(self.global_model),
                avg_local_loss=sum(losses)/len(losses),
                communication_bytes=bytes_communicated
            ))
        return self.history

# Simulate heterogeneous clients (non-IID)
client_data_means = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
clients = [FLClient(i, n_samples=rng, data_mean=m, data_std=1.0)
           for i, (rng, m) in enumerate(zip(
               [random.randint(100,1000) for _ in range(10)], client_data_means))]

fl = FedAvgSimulator(clients, global_model=[0.0], n_rounds=15, fraction_clients=0.5)
history = fl.run()
print(f'FedAvg over {len(history)} rounds:')
print(f'{'Round':>7} {'Clients':>8} {'Global Model':>14} {'Avg Local Loss':>15}')
print('-' * 50)
for r in history[::3]:
    print(f'{r.round_num:>7} {r.n_clients:>8} {r.global_model[0]:>14.4f} {r.avg_local_loss:>15.6f}')
print(f'\nTrue global mean: {sum(client_data_means)/len(client_data_means):.4f}')
print(f'Final model value: {history[-1].global_model[0]:.4f}')

## Non-IID Convergence Challenges

When client data distributions differ significantly, FedAvg's weighted average can diverge from the optimal global model. Client drift — local models moving toward local optima during local epochs — is the primary cause.

**FedProx** (next notebook) addresses this by adding a proximal term that keeps local models close to the global model during local training.

**SCAFFOLD** uses variance reduction to correct for client drift using control variates.

The severity of the non-IID problem depends on the degree of heterogeneity. For production federated systems, monitoring per-client accuracy and detecting outlier clients is essential for quality control.